# Cloud Type Baseline

In [ ]:
# !pip install pytorch-lightning lightning segmentation_models_pytorch yacs deepspeed xarray

In [ ]:
import os
import sys

sys.path.append('/explore/nobackup/people/jacaraba/development/satvision-pix4d')

# os.environ['MASTER_PORT'] = '0'

import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

import torchvision.models as models
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp
import tqdm
import pandas as pd

from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    RichProgressBar,
    TQDMProgressBar
)
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassJaccardIndex,
)

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from satvision_pix4d.configs.config import _C, _update_config_from_file
from satvision_pix4d.utils import get_strategy, get_distributed_train_batches
from satvision_pix4d.pipelines import PIPELINES, get_available_pipelines
from satvision_pix4d.datamodules import DATAMODULES, get_available_datamodules
from satvision_pix4d.models.encoders.mae import build_satmae_model
from satvision_pix4d.datasets.abi_temporal_benchmark_dataset import ABITemporalBenchmarkDataset

In [ ]:
DATA_DIRECTORY = "/explore/nobackup/projects/pix4dcloud/szhang16/abiChips/GOES-16"
WORK_DIR = "/explore/nobackup/projects/ilab/projects/SatVisionPix4D/downstream_tasks/cloud_types/cloud_types_baseline"
SATVISION_PIX4D_MODEL = "/explore/nobackup/projects/ilab/projects/SatVisionPix4D/pretraining/mp_rank_00_model_states.pt"
SATVISION_PIX4D_CONFIG = "/explore/nobackup/projects/ilab/projects/SatVisionPix4D/pretraining/test_satmae_dev_dgx.yaml"
BATCH_SIZE = 256
NUM_WORKERS = 40
MAX_EPOCHS = 6000
LEARNING_RATE = 1e-4
TARGET_HEIGHT = 96
TARGET_WIDTH = 40
NUM_SAMPLES_TO_TEST = 5

pl.seed_everything(42)

In [ ]:
config = _C.clone()
_update_config_from_file(config, SATVISION_PIX4D_CONFIG)
print("Loaded configuration file.")

In [ ]:
# Add checkpoint (MODEL.PRETRAINED), 
# validation tile dir (DATA.DATA_PATHS),
# and output dir (OUTPUT) to config file
config.defrost()
config.MODEL.PRETRAINED = SATVISION_PIX4D_MODEL
config.OUTPUT = '.'
config.freeze()
print("Updated configuration file.")

# Get the proper pipeline
available_pipelines = get_available_pipelines()
print("Available pipelines:", available_pipelines)

pipeline = PIPELINES[config.PIPELINE]
print(f'Using {pipeline}')

ptlPipeline = pipeline(config)

# Resume from checkpoint
print(f'Attempting to resume from checkpoint {config.MODEL.RESUME}')
sat_pretrain = ptlPipeline.load_checkpoint(config.MODEL.PRETRAINED, config)
print(sat_pretrain)

pe = sat_pretrain.model.patch_embed
if hasattr(pe, "strict_img_size"):
    pe.strict_img_size = False
if hasattr(pe, "img_size"):
    pe.img_size = None
if hasattr(pe, "dynamic_img_pad"):
    pe.dynamic_img_pad = True

print('Successfully applied checkpoint')

print("Loaded SatVision pretrain:", type(sat_pretrain))
print("Underlying MAE:", type(sat_pretrain.model))

In [ ]:
class SimpleSegDecoder(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_dim, 256, 1),
            nn.GELU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, feat_map, out_hw):
        x = self.block(feat_map)
        x = F.interpolate(x, size=out_hw, mode="bilinear", align_corners=False)
        return x


class SatMAEToSeg(nn.Module):
    """
    Wraps SatVision Pix4D MAE pretrain checkpoint and produces segmentation logits.
    """
    def __init__(self, ptl_pretrain_model, num_classes: int, patch_size: int = 16, freeze_encoder: bool = True):
        super().__init__()
        self.ptl = model                  # SatVisionPix4DSatMAEPretrain
        self.mae = model.model            # MaskedAutoencoderViT
        self.patch_size = patch_size
        self.embed_dim = 1024                          # from summary (16->1024 patch embed conv)
        self.decoder = SimpleSegDecoder(self.embed_dim, num_classes)

        if freeze_encoder:
            for p in self.mae.parameters():
                p.requires_grad = False

    def _tokens_from_encoder(self, imgs, ts):
        # Preferred path
        if hasattr(self.mae, "forward_encoder"):
            out = self.mae.forward_encoder(imgs, ts, mask_ratio=0.0)
            # common patterns: (latent, mask, ids_restore) or similar
            if isinstance(out, (tuple, list)):
                latent = out[0]
            else:
                latent = out
            return latent  # (B, N, D)

        # Fallback path: use wrapper forward and interpret pred as tokens
        out = self.ptl(imgs, ts)
        if isinstance(out, (tuple, list)) and len(out) >= 2:
            pred = out[1]
            if pred.ndim == 3:
                return pred  # (B, N, D)
        raise RuntimeError("Could not extract encoder tokens; inspect MAE forward methods.")

    def forward(self, imgs, ts):
        """
        imgs: (B, T, C, H, W)
        ts:   (B, T, 3)
        returns logits: (B, K, H, W)
        """
        B, T, C, H, W = imgs.shape
        assert H % self.patch_size == 0 and W % self.patch_size == 0, "H/W must be divisible by patch_size=16"

        tokens = self._tokens_from_encoder(imgs, ts)      # (B, N, D)
        # Some encoders include a CLS token; if so, drop it
        if tokens.shape[1] == (T * (H // self.patch_size) * (W // self.patch_size) + 1):
            tokens = tokens[:, 1:, :]

        h = H // self.patch_size
        w = W // self.patch_size
        N_expected = T * h * w
        if tokens.shape[1] != N_expected:
            raise RuntimeError(f"Token count mismatch: got {tokens.shape[1]}, expected {N_expected} (=T*h*w)")

        # (B, N, D) -> (B, T, h, w, D) -> aggregate over time -> (B, D, h, w)
        tokens = tokens.view(B, T, h, w, self.embed_dim)
        # feat_map = tokens.mean(dim=1).permute(0, 4, 2, 3).contiguous()  # (B, D, h, w)
        feat_map = tokens.mean(dim=1).permute(0, 3, 1, 2).contiguous()  # (B, D, h, w)

        logits = self.decoder(feat_map, out_hw=(H, W))  # (B, K, H, W)
        return logits

import torch
import torch.nn as nn
import torch.nn.functional as F

class SatMAEToSegFixed(nn.Module):
    def __init__(
        self,
        ptl_pretrain_model,      # SatVisionPix4DSatMAEPretrain LightningModule
        num_classes=9,
        patch_size=16,
        output_shape=(96, 40),
        freeze_encoder=True,
    ):
        super().__init__()
        self.ptl = ptl_pretrain_model
        self.mae = ptl_pretrain_model.model  # MaskedAutoencoderViT
        self.patch_size = patch_size
        self.embed_dim = 1024                # from your checkpoint summary
        self.output_shape = output_shape

        # Simple decoder from (B,1024,8,8) -> (B,num_classes,128,128)
        # (You can swap this for a bigger decoder later.)
        self.decoder = nn.Sequential(
            nn.Conv2d(self.embed_dim, 512, 1),
            nn.GELU(),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(256, num_classes, 1),
        )

        # Force final output size like your UNet
        self.final_upsample = nn.Upsample(size=output_shape, mode="bilinear", align_corners=False)

        if freeze_encoder:
            self.freeze_encoder()

    def freeze_encoder(self):
        for p in self.mae.parameters():
            p.requires_grad = False
    
    def unfreeze_encoder(self):
        for p in self.mae.parameters():
            p.requires_grad = True
    
    def _encode_tokens(self, imgs_btchw, ts_bt3):
        if not hasattr(self.mae, "forward_encoder"):
            raise RuntimeError(
                "Expected SatMAE model.model to implement forward_encoder(imgs, ts, mask_ratio). "
                f"Got type={type(self.mae)}"
            )
        out = self.mae.forward_encoder(imgs_btchw, ts_bt3, mask_ratio=0.0)
        latent = out[0] if isinstance(out, (tuple, list)) else out
        return latent


    def forward(self, chips_bchw):
        """
        chips_bchw: (B, 16, 128, 128)
        returns logits: (B, 9, 96, 40)
        """
        B, C, H, W = chips_bchw.shape
        P = self.patch_size
        assert H % P == 0 and W % P == 0, f"Input H,W must be divisible by patch_size={P}"

        # Add time dimension for SatMAE: (B,1,C,H,W)
        imgs = chips_bchw.unsqueeze(1)

        # Dummy timestamps (B,1,3)
        ts = torch.zeros((B, 1, 3), device=chips_bchw.device, dtype=torch.float32)

        tokens = self._encode_tokens(imgs, ts)  # expected (B, N, D)
        if tokens.ndim != 3:
            raise RuntimeError(f"Expected tokens (B,N,D), got {tokens.shape}")

        h = H // P
        w = W // P
        N_expected = 1 * h * w

        # Drop CLS token if present
        if tokens.shape[1] == N_expected + 1:
            tokens = tokens[:, 1:, :]

        if tokens.shape[1] != N_expected:
            raise RuntimeError(f"Token count mismatch: got {tokens.shape[1]}, expected {N_expected} (h*w={h*w})")

        # (B, N, D) -> (B, h, w, D) -> (B, D, h, w)
        feat_map = tokens.view(B, h, w, self.embed_dim).permute(0, 3, 1, 2).contiguous()  # (B,1024,8,8)

        # Decode to logits at patch-grid then upsample to full 128x128
        logits = self.decoder(feat_map)  # (B,K,8,8)
        logits = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)  # (B,K,128,128)

        # Final forced output shape (96,40) like your baseline UNet
        logits = self.final_upsample(logits)  # (B,K,96,40)
        return logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x): return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_ch, out_ch)

    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        return x

class UNetDecoderNoSkips(nn.Module):
    """
    Input: (B, D, 8, 8)  -> Output: (B, K, 128, 128)
    """
    def __init__(self, in_dim=1024, num_classes=9, base=256):
        super().__init__()
        # compress first so decoder isn't huge
        self.stem = DoubleConv(in_dim, base)          # 1024 -> 256 at 8x8
        self.up1  = UpBlock(base, base//2)            # 8 -> 16
        self.up2  = UpBlock(base//2, base//4)         # 16 -> 32
        self.up3  = UpBlock(base//4, base//8)         # 32 -> 64
        self.up4  = UpBlock(base//8, base//16)        # 64 -> 128
        self.head = nn.Conv2d(base//16, num_classes, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x)
        return self.head(x)

class SatMAEToSegFixedUNet(nn.Module):
    def __init__(
        self,
        ptl_pretrain_model,      # SatVisionPix4DSatMAEPretrain LightningModule
        num_classes=9,
        patch_size=16,
        output_shape=(96, 40),
        freeze_encoder=True,
    ):
        super().__init__()
        self.ptl = ptl_pretrain_model
        self.mae = ptl_pretrain_model.model  # MaskedAutoencoderViT
        self.patch_size = patch_size
        self.embed_dim = 1024                # from your checkpoint summary
        self.output_shape = output_shape

        # Simple decoder from (B,1024,8,8) -> (B,num_classes,128,128)
        # (You can swap this for a bigger decoder later.)
        self.decoder = nn.Sequential(
            nn.Conv2d(self.embed_dim, 512, 1),
            nn.GELU(),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(256, num_classes, 1),
        )
        self.decoder = UNetDecoderNoSkips(in_dim=self.embed_dim, num_classes=num_classes, base=256)


        # Force final output size like your UNet
        self.final_upsample = nn.Upsample(size=output_shape, mode="bilinear", align_corners=False)

        if freeze_encoder:
            self.freeze_encoder()

    def freeze_encoder(self):
        for p in self.mae.parameters():
            p.requires_grad = False
    
    def unfreeze_encoder(self):
        for p in self.mae.parameters():
            p.requires_grad = True
    
    def _encode_tokens(self, imgs_btchw, ts_bt3):
        if not hasattr(self.mae, "forward_encoder"):
            raise RuntimeError(
                "Expected SatMAE model.model to implement forward_encoder(imgs, ts, mask_ratio). "
                f"Got type={type(self.mae)}"
            )
        out = self.mae.forward_encoder(imgs_btchw, ts_bt3, mask_ratio=0.0)
        latent = out[0] if isinstance(out, (tuple, list)) else out
        return latent


    def forward(self, chips_bchw):
        """
        chips_bchw: (B, 16, 128, 128)
        returns logits: (B, 9, 96, 40)
        """
        B, C, H, W = chips_bchw.shape
        P = self.patch_size
        assert H % P == 0 and W % P == 0, f"Input H,W must be divisible by patch_size={P}"

        # Add time dimension for SatMAE: (B,1,C,H,W)
        imgs = chips_bchw.unsqueeze(1)

        # Dummy timestamps (B,1,3)
        ts = torch.zeros((B, 1, 3), device=chips_bchw.device, dtype=torch.float32)

        tokens = self._encode_tokens(imgs, ts)  # expected (B, N, D)
        if tokens.ndim != 3:
            raise RuntimeError(f"Expected tokens (B,N,D), got {tokens.shape}")

        h = H // P
        w = W // P
        N_expected = 1 * h * w

        # Drop CLS token if present
        if tokens.shape[1] == N_expected + 1:
            tokens = tokens[:, 1:, :]

        if tokens.shape[1] != N_expected:
            raise RuntimeError(f"Token count mismatch: got {tokens.shape[1]}, expected {N_expected} (h*w={h*w})")

        # (B, N, D) -> (B, h, w, D) -> (B, D, h, w)
        feat_map = tokens.view(B, h, w, self.embed_dim).permute(0, 3, 1, 2).contiguous()  # (B,1024,8,8)

        # Decode to logits at patch-grid then upsample to full 128x128
        #logits = self.decoder(feat_map)  # (B,K,8,8)
        #logits = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)  # (B,K,128,128)
        logits = self.decoder(feat_map)          # now returns (B,K,128,128)
        logits = self.final_upsample(logits)     # -> (B,K,96,40)

        # Final forced output shape (96,40) like your baseline UNet
        #logits = self.final_upsample(logits)  # (B,K,96,40)
        return logits

In [ ]:
# 1. Create dummy dataset
#dummy_dataset = ABITemporalBenchmarkDataset(
#    data_paths=[],
#    split="val",
#    img_size=512,
#    in_chans=16,
#    num_timesteps=7
#)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sat_seg = SatMAEToSegFixed(sat_pretrain, num_classes=9, output_shape=(96,40), freeze_encoder=True).cuda()#.eval()
sat_seg

In [ ]:
"""
class CloudSatDataset(Dataset):
    def __init__(self, file_paths, target_size=(96, 40)):
        self.file_paths = file_paths
        self.target_height, self.target_width = target_size


    def __len__(self):
        return len(self.file_paths)


    def __getitem__(self, idx):
        with np.load(self.file_paths[idx], allow_pickle=True) as data:
            abi_chip = data['chip'].astype(np.float32)

            for i in range(abi_chip.shape[2]):
                channel = abi_chip[:, :, i]
                min_val = channel.min()
                max_val = channel.max()
                if max_val > min_val:
                    abi_chip[:, :, i] = (channel - min_val)/(max_val - min_val)
            
            abi_chip = np.transpose(abi_chip, (2, 0, 1))

            cloud_mask_raw = data['data'].item()['Cloud_mask'].astype(np.int64)
            cloud_mask_raw = cloud_mask_raw[:, :34]
            pad_height_total = self.target_height - cloud_mask_raw.shape[0]
            pad_top = pad_height_total // 2
            pad_bottom = pad_height_total - pad_top
            pad_width_total = self.target_width - cloud_mask_raw.shape[1]
            pad_left = pad_width_total // 2
            pad_right = pad_width_total - pad_left
            cloud_mask_padded = np.pad(
                cloud_mask_raw,
                ((pad_top, pad_bottom), (pad_left, pad_right)),
                'constant',
                constant_values=0
            )
           
            return {
                "chip": torch.from_numpy(abi_chip),
                "mask": torch.from_numpy(cloud_mask_padded),
                "path": self.file_paths[idx]
            }
"""
from torch.utils.data import Dataset
import numpy as np
import torch

class CloudSatDataset(Dataset):
    def __init__(
        self,
        file_paths,
        target_size=(96, 40),
        ignore_index=255,
        ignore_not_determined=False,  # set False if you truly want class 0 as a supervised class
    ):
        self.file_paths = file_paths
        self.target_height, self.target_width = target_size
        self.ignore_index = int(ignore_index)
        self.ignore_not_determined = bool(ignore_not_determined)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with np.load(self.file_paths[idx], allow_pickle=True) as data:
            # --- Chip: (H, W, C) -> normalize each channel -> (C, H, W) ---
            abi_chip = data["chip"].astype(np.float32)

            # per-channel min/max normalization
            for i in range(abi_chip.shape[2]):
                channel = abi_chip[:, :, i]
                mn = channel.min()
                mx = channel.max()
                if mx > mn:
                    abi_chip[:, :, i] = (channel - mn) / (mx - mn)
                else:
                    abi_chip[:, :, i] = 0.0  # constant channel

            abi_chip = np.transpose(abi_chip, (2, 0, 1))  # (C, H, W)

            # --- Mask: crop + pad to (target_height, target_width) ---
            cloud_mask_raw = data["data"].item()["Cloud_mask"].astype(np.int64)

            # your existing crop
            cloud_mask_raw = cloud_mask_raw[:, :34]  # -> (H_raw, W_raw)

            # Optionally ignore "Not Determined" label everywhere
            if self.ignore_not_determined:
                cloud_mask_raw = cloud_mask_raw.copy()
                cloud_mask_raw[cloud_mask_raw == 0] = self.ignore_index

            h_raw, w_raw = cloud_mask_raw.shape

            # If raw is bigger than target, center-crop it (safer than negative padding)
            if h_raw > self.target_height:
                top = (h_raw - self.target_height) // 2
                cloud_mask_raw = cloud_mask_raw[top : top + self.target_height, :]
                h_raw = self.target_height

            if w_raw > self.target_width:
                left = (w_raw - self.target_width) // 2
                cloud_mask_raw = cloud_mask_raw[:, left : left + self.target_width]
                w_raw = self.target_width

            pad_h = self.target_height - h_raw
            pad_w = self.target_width - w_raw

            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left

            cloud_mask_padded = np.pad(
                cloud_mask_raw,
                ((pad_top, pad_bottom), (pad_left, pad_right)),
                mode="constant",
                constant_values=self.ignore_index,
            ).astype(np.int64)

            return {
                "chip": torch.from_numpy(abi_chip),               # float32
                "mask": torch.from_numpy(cloud_mask_padded),      # int64
                "path": self.file_paths[idx],
            }


In [ ]:
class CloudSatDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=16, num_workers=0, train_val_test_split=(0.8, 0.1, 0.1)):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_val_test_split = train_val_test_split
        self.file_paths = sorted(glob.glob(os.path.join(self.data_dir, '*.npz')))

    def setup(self, stage=None):
        n_total = len(self.file_paths)
        n_train = int(n_total * self.train_val_test_split[0])
        n_val = int(n_total * self.train_val_test_split[1])
       
        self.train_files = self.file_paths[:n_train]
        self.val_files = self.file_paths[n_train : n_train + n_val]
        self.test_files = self.file_paths[n_train + n_val :]


        self.train_dataset = CloudSatDataset(self.train_files)
        self.val_dataset = CloudSatDataset(self.val_files)
        self.test_dataset = CloudSatDataset(self.test_files)


    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, batch_size=self.batch_size,
            shuffle=True, num_workers=self.num_workers,
            pin_memory=False, collate_fn=self._collate_fn
        )
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, batch_size=self.batch_size,
            num_workers=self.num_workers, pin_memory=False,
            collate_fn=self._collate_fn
        )
    def test_dataloader(self):
        return DataLoader(
            self.test_dataset, batch_size=self.batch_size,
            num_workers=self.num_workers, pin_memory=False,
            collate_fn=self._collate_fn
        )

    def _collate_fn(self, batch):
        batch = list(filter(lambda x: x is not None, batch))
        if not batch:
            return None
        return torch.utils.data.dataloader.default_collate(batch)

    def debug_shapes(self):
        loader = self.train_dataloader()
        batch = next(iter(loader))
        if batch is None:
            print("Empty batch")
            return
    
        print("=== DEBUG: Train batch shapes ===")
        for k, v in batch.items():
            if torch.is_tensor(v):
                print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")
            else:
                print(f"{k}: type={type(v)}")

In [ ]:
datamodule = CloudSatDataModule(data_dir=DATA_DIRECTORY, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
datamodule.setup()
datamodule.debug_shapes()

In [ ]:
class CustomUNET(nn.Module):
    def __init__(self, in_channels=16, num_classes=9, output_shape=(96, 40)):
        super().__init__()
        self.layers = [in_channels, 64, 128, 256, 512, 1024]

        self.double_conv_downs = nn.ModuleList(
            [self.__double_conv(layer, layer_n) for layer, layer_n in zip(self.layers[:-1], self.layers[1:])])

        self.up_trans = nn.ModuleList(
            [nn.ConvTranspose2d(layer, layer_n, kernel_size=2, stride=2)
             for layer, layer_n in zip(self.layers[::-1][:-2], self.layers[::-1][1:-1])])

        self.double_conv_ups = nn.ModuleList(
            [self.__double_conv(layer, layer//2) for layer in self.layers[::-1][:-2]])

        self.max_pool_2x2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

        self.final_upsample = nn.Upsample(
            size=output_shape, mode='bilinear', align_corners=False
        )

    def __double_conv(self, in_channels, out_channels):
        conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        return conv

    def forward(self, x):
        # Down layers
        concat_layers = []
        for down in self.double_conv_downs:
            x = down(x)
            if down != self.double_conv_downs[-1]:
                concat_layers.append(x)
                x = self.max_pool_2x2(x)
        
        concat_layers = concat_layers[::-1]

        # Up layers
        for up_trans, double_conv_up, concat_layer in zip(self.up_trans, self.double_conv_ups, concat_layers):
            x = up_trans(x)
            if x.shape != concat_layer.shape:
                x = TF.resize(x, concat_layer.shape[2:])
            concatenated = torch.cat((concat_layer, x), dim=1)
            x = double_conv_up(concatenated)

        x = self.final_conv(x)
        x = self.final_upsample(x)
        return x

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth


    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        probs = F.softmax(logits, dim=1)
       
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dims)
        cardinality = torch.sum(probs + targets_one_hot, dims)
       
        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1. - dice_score.mean()

import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedDiceLoss(nn.Module):
    def __init__(self, num_classes: int, ignore_index: int = 255, eps: float = 1e-6):
        super().__init__()
        self.num_classes = int(num_classes)
        self.ignore_index = int(ignore_index)
        self.eps = float(eps)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits:  (B, C, H, W) float
        targets: (B, H, W) long, may contain ignore_index
        """
        B, C, H, W = logits.shape
        assert C == self.num_classes, f"logits C={C} != num_classes={self.num_classes}"

        targets = targets.long()

        # valid pixels mask
        valid = (targets != self.ignore_index)  # (B,H,W)

        # replace ignored targets with 0 so one_hot/scatter is safe;
        # we will zero them out using the valid mask anyway.
        targets_safe = targets.clone()
        targets_safe[~valid] = 0

        # probs: (B,C,H,W)
        probs = torch.softmax(logits, dim=1)

        # one-hot: (B,H,W,C) -> (B,C,H,W)
        onehot = F.one_hot(targets_safe, num_classes=C).permute(0, 3, 1, 2).float()

        # apply mask
        valid_f = valid.unsqueeze(1).float()  # (B,1,H,W)
        probs = probs * valid_f
        onehot = onehot * valid_f

        # dice per class over batch+spatial
        dims = (0, 2, 3)
        intersection = (probs * onehot).sum(dims)
        union = probs.sum(dims) + onehot.sum(dims)

        dice = (2.0 * intersection + self.eps) / (union + self.eps)

        # If a class is completely absent in the valid region, union==0 -> dice==1 by eps;
        # that's fine, or you can drop absent classes. Leaving as-is is standard.
        loss = 1.0 - dice.mean()
        return loss


In [ ]:
class CustomUNETLightning(pl.LightningModule):
    def __init__(
        self,
        backbone: nn.Module | None = None,
        in_channels=16,
        num_classes=9,
        target_height=96,
        target_width=40,
        lr=1e-4,
        dice_weight=0.5,
        ignore_index=None,
        average="macro",
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["backbone"])

        # <-- swap model here
        if backbone is None:
            self.model = CustomUNET(
                in_channels=in_channels,
                num_classes=num_classes,
                output_shape=(target_height, target_width),
            )
        else:
            self.model = backbone  # must output (B, num_classes, target_height, target_width)

        #self.ce_loss = nn.CrossEntropyLoss(ignore_index=ignore_index if ignore_index is not None else -100)
        #self.dice_loss = DiceLoss()
        #self.dice_weight = dice_weight

        ignore_index = 255
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.dice_loss = MaskedDiceLoss(num_classes=num_classes, ignore_index=ignore_index)
        self.dice_weight = dice_weight
    
        self.train_acc = MulticlassAccuracy(num_classes=num_classes, average=average)
        self.val_acc   = MulticlassAccuracy(num_classes=num_classes, average=average)
        self.test_acc  = MulticlassAccuracy(num_classes=num_classes, average=average)

        self.train_f1 = MulticlassF1Score(num_classes=num_classes, average=average)
        self.val_f1   = MulticlassF1Score(num_classes=num_classes, average=average)
        self.test_f1  = MulticlassF1Score(num_classes=num_classes, average=average)

        self.train_iou = MulticlassJaccardIndex(num_classes=num_classes, average=average)
        self.val_iou   = MulticlassJaccardIndex(num_classes=num_classes, average=average)
        self.test_iou  = MulticlassJaccardIndex(num_classes=num_classes, average=average)

        if ignore_index is not None:
            for m in [self.train_acc, self.val_acc, self.test_acc,
                      self.train_f1, self.val_f1, self.test_f1,
                      self.train_iou, self.val_iou, self.test_iou]:
                if hasattr(m, "ignore_index"):
                    m.ignore_index = ignore_index

        self.unfreeze_encoder_at_epoch = 2   # pick 0/1/2 etc
        self.encoder_lr = 1e-5              # smaller LR for backbone
        self.head_lr = lr                   # your existing lr for decoder/head
        self._encoder_unfrozen = False

    
    def forward(self, x):
        return self.model(x)

    def _common_step(self, batch, batch_idx, stage: str):
        if batch is None:
            return None

        chips, masks = batch["chip"], batch["mask"]  # masks: [B, H, W] with class ids
        logits = self.forward(chips)                 # logits: [B, C, H, W]

        ce = self.ce_loss(logits, masks.long())
        dice = self.dice_loss(logits, masks.long())
        loss = self.dice_weight * dice + (1 - self.dice_weight) * ce

        preds = torch.argmax(logits, dim=1)          # [B, H, W]
        bs = chips.size(0)

        # ---- NEW: mask out ignore pixels for metrics ----
        ignore_index = self.ce_loss.ignore_index  # 255 in your case
        valid = (masks != ignore_index)

        # flatten to 1D vectors of valid pixels
        preds_v = preds[valid]
        masks_v = masks[valid]

        # If a whole batch is ignored (rare but possible), skip metric update
        if preds_v.numel() == 0:
            return loss
        
        # Update metrics
        if stage == "train":
            self.train_acc.update(preds, masks)
            self.train_f1.update(preds, masks)
            self.train_iou.update(preds, masks)
        elif stage == "val":
            self.val_acc.update(preds, masks)
            self.val_f1.update(preds, masks)
            self.val_iou.update(preds, masks)
        elif stage == "test":
            self.test_acc.update(preds, masks)
            self.test_f1.update(preds, masks)
            self.test_iou.update(preds, masks)

        # Log losses (show total loss in prog bar)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=True if stage == "train" else False, on_epoch=True, batch_size=bs, sync_dist=(stage != "train"))
        self.log(f"{stage}_ce_loss", ce, prog_bar=False, on_step=False, on_epoch=True, batch_size=bs, sync_dist=(stage != "train"))
        self.log(f"{stage}_dice_loss", dice, prog_bar=False, on_step=False, on_epoch=True, batch_size=bs, sync_dist=(stage != "train"))

        # Log metrics (epoch-level; put these in progress bar)
        if stage == "train":
            self.log("train_acc", self.train_acc, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs)
            self.log("train_f1",  self.train_f1,  prog_bar=True, on_step=False, on_epoch=True, batch_size=bs)
            self.log("train_iou", self.train_iou, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs)
        elif stage == "val":
            self.log("val_acc", self.val_acc, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("val_f1",  self.val_f1,  prog_bar=True, on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("val_iou", self.val_iou, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
        elif stage == "test":
            self.log("test_acc", self.test_acc, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("test_f1",  self.test_f1,  prog_bar=True, on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)
            self.log("test_iou", self.test_iou, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs, sync_dist=True)

        return loss

    def on_train_epoch_start(self):
        if (
            (not self._encoder_unfrozen)
            and (self.current_epoch >= self.unfreeze_encoder_at_epoch)
            and hasattr(self.model, "unfreeze_encoder")
        ):
            self.model.unfreeze_encoder()
            self._encoder_unfrozen = True
            print(f"[INFO] Unfroze encoder at epoch {self.current_epoch}")

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "val")

    def test_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, "test")

    #def configure_optimizers(self):
    #    return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
    def configure_optimizers(self):
        # If this is SatMAEToSegFixed, it has self.mae (encoder) and self.decoder (head)
        if hasattr(self.model, "mae") and hasattr(self.model, "decoder"):
            enc_params = [p for p in self.model.mae.parameters()]
            head_params = [p for p in self.model.decoder.parameters()] + \
                          [p for p in self.model.final_upsample.parameters()]
    
            opt = torch.optim.AdamW(
                [
                    {"params": enc_params, "lr": self.encoder_lr},
                    {"params": head_params, "lr": self.head_lr},
                ],
                weight_decay=1e-4,
            )
            return opt
    
        # fallback (e.g., baseline UNet)
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)


In [ ]:
# --- 4. Visualization Function ---
def visualize_prediction(chip_tensor, true_mask_padded, pred_mask_padded, file_path):
    chip = chip_tensor[0].cpu().numpy()
    true_mask = true_mask_padded.cpu().numpy()
    pred_mask = pred_mask_padded.cpu().numpy()
   
    original_height, original_width = 91, 34
    pad_height_total = true_mask.shape[0] - original_height
    pad_top = pad_height_total // 2
    pad_width_total = true_mask.shape[1] - original_width
    pad_left = pad_width_total // 2
   
    true_mask_unpadded = true_mask[pad_top : pad_top + original_height, pad_left : pad_left + original_width]
    pred_mask_unpadded = pred_mask[pad_top : pad_top + original_height, pad_left : pad_left + original_width]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6))
    fig.suptitle(f"File: {os.path.basename(file_path)}", fontsize=16)

    ax1.imshow(chip, cmap='gray')
    ax1.set_title("Input ABI Chip (Channel 1)")
    ax1.axis('off')

    plot_curtain(ax2, true_mask_unpadded, "Ground Truth Classification", fig)
    plot_curtain(ax3, pred_mask_unpadded, "Predicted Classification", fig)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    output_filename = os.path.basename(file_path).replace('.npz', '.png')
    save_dir = os.path.join(WORK_DIR, 'predictions')
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, output_filename)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    #plt.close(fig)
    print(f"Saved visualization to: {save_path}")
    plt.show()

def plot_curtain(ax, mask, title, fig):
    num_classes = 9
    zz = mask.T
   
    x_axis_points = np.arange(zz.shape[1])
    y_axis_km = np.arange(0, zz.shape[0] * 0.5, 0.5)


    mesh = ax.pcolormesh(x_axis_points, y_axis_km, zz, cmap='tab10', shading='nearest', vmin=-0.5, vmax=num_classes - 0.5)
   
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Point Along Track", fontsize=12)
    ax.set_ylabel("Height (km)", fontsize=12)
    ax.set_ylim(0, zz.shape[0] * 0.5)
    ax.grid(True, linestyle='--', alpha=0.5)
   
    cbar = fig.colorbar(mesh, ax=ax, ticks=np.arange(num_classes))
    cbar.set_ticklabels(['0: Not Determined', '1: Cirrus', '2: Alotstratus', '3: Altocumulus', '4: Stratus', '5: Stratocumulus', '6: Cumulus', '7: Nimbostratus', '8: Deep Convection'])



def calculate_iou(pred_mask, true_mask, num_classes):
    
    pred_mask = pred_mask.flatten()
    true_mask = true_mask.flatten()
    
    per_class_iou = np.zeros(num_classes)

    for cls in range(num_classes):
        intersection = np.sum((pred_mask == cls) & (true_mask == cls))
        
        union = np.sum((pred_mask == cls) | (true_mask == cls))
        
        if union == 0:
            per_class_iou[cls] = np.nan
        else:
            per_class_iou[cls] = intersection / union
            
    mIoU = np.nanmean(per_class_iou)
    
    return mIoU, per_class_iou

In [ ]:
datamodule = CloudSatDataModule(data_dir=DATA_DIRECTORY, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

In [ ]:
sat_seg_backbone = SatMAEToSegFixedUNet(
    ptl_pretrain_model=sat_pretrain,
    num_classes=9,
    patch_size=16,
    output_shape=(96, 40),
    freeze_encoder=True,
)


lit = CustomUNETLightning(
    backbone=sat_seg_backbone,
    in_channels=16,
    num_classes=9,
    target_height=96,
    target_width=40,
    lr=1e-4,
    dice_weight=0.5,   # first-pass recommendation
    ignore_index=255,
)


In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath=os.path.join(WORK_DIR, 'checkpoints'),
    filename='cloudsat-best-model-{epoch:02d}-{val_loss:.2f}',
    save_top_k=1,
    mode='min',
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=True,
    mode='min'
)

In [ ]:
trainer = pl.Trainer(
    callbacks=[checkpoint_callback, early_stop_callback, TQDMProgressBar(refresh_rate=20)],
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    log_every_n_steps=10,
    default_root_dir=WORK_DIR
)
trainer.fit(lit, datamodule)


## Inference

In [ ]:
best_model_path = checkpoint_callback.best_model_path
print(f"\n--- Loading best model for evaluation: {os.path.basename(best_model_path)} ---")

In [ ]:
#trained_model = CustomUNETLightning.load_from_checkpoint(best_model_path)
#trained_model.eval()

#backbone = SatMAEToSegFixed(...)  # same config as training
trained_model = CustomUNETLightning.load_from_checkpoint(
    best_model_path,
    backbone=sat_seg_backbone,
    in_channels=16,
    num_classes=9,
    target_height=96,
    target_width=40,
)
trained_model.eval()


In [ ]:
# Infer model device + dtype from first parameter
#p = next(trained_model.parameters())
#model_device = p.device
#model_dtype = p.dtype
#device = next(trained_model.parameters()).device
#print("Model device:", model_device, "Model dtype:", model_dtype)

In [ ]:
datamodule.setup('test')
test_loader = datamodule.test_dataloader()

In [ ]:
all_preds = []
all_true_masks = []

print(f"\n--- Evaluating model on the full test set ({len(datamodule.test_files)} samples) ---")
with torch.no_grad():
    for batch in tqdm.tqdm(test_loader, desc="Testing"):
        if batch is None:
            continue
        chips = batch['chip'].float().to(device, dtype=torch.float32, non_blocking=True)
        true_masks = batch['mask'].long().to(device, dtype=torch.long, non_blocking=True)
        
        logits = trained_model(chips)
        predicted_masks = torch.argmax(logits, dim=1)
        
        all_preds.append(predicted_masks.cpu().numpy())
        all_true_masks.append(true_masks.cpu().numpy())

if all_preds:
        all_preds = np.concatenate(all_preds, axis=0)
        all_true_masks = np.concatenate(all_true_masks, axis=0)

        NUM_CLASSES = 9
        CLASS_NAMES = ['0: Not Determined', '1: Cirrus', '2: Alotstratus', '3: Altocumulus', '4: Stratus', '5: Stratocumulus', '6: Cumulus', '7: Nimbostratus', '8: Deep Convection']

        mean_iou, class_iou = calculate_iou(all_preds, all_true_masks, NUM_CLASSES)
        
        # --- Print Results Table ---
        iou_data = {'Class Name': CLASS_NAMES, 'IoU': class_iou}
        iou_df = pd.DataFrame(iou_data)
        
        print("\n--- Model Performance on Test Set ---")
        print(f"\nMean IoU (mIoU): {mean_iou:.4f}\n")
        print("Per-Class IoU:")
        print(iou_df.to_string(index=False, float_format="%.4f"))

vis_loader = DataLoader(datamodule.test_dataset, batch_size=5, shuffle=True, collate_fn=datamodule._collate_fn)
print(f"\n--- Visualizing {5} random test samples ---")
vis_batch = next(iter(vis_loader))
if vis_batch:
    vis_chips = vis_batch['chip'].float().to(device, dtype=torch.float32, non_blocking=True)
    vis_true_masks = vis_batch['mask'].long().to(device, dtype=torch.long, non_blocking=True)
    vis_file_paths = vis_batch['path']

    with torch.no_grad():
        vis_logits = trained_model(vis_chips)
        vis_predicted_masks = torch.argmax(vis_logits, dim=1)

    for i in range(len(vis_chips)):
        visualize_prediction(vis_chips[i], vis_true_masks[i], vis_predicted_masks[i], vis_file_paths[i])